# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitttt077/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Author:** Harshit Kudhial (`@harshitttt077`)  
**Track:** Machine Learning · Foundations (Week 3)  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Date:** September 2026  

---

## 1. The Data Contract in Plain Words (5 Answers)

1. **What One Row Means:** Exactly one published web page / URL asset (`content_id`) belonging to an enterprise domain (`client_id`).
2. **Which Table(s) We Use:** `fact_content_daily_performance` joined with `dim_content` from the FlyRank Search Intelligence Warehouse (mirrored locally in `data/raw/content_refresh_anonymized.csv`).
3. **Which Time Window:** Trailing 90-day aggregation window for pre-decision historical search metrics (impressions, clicks, position, engagement).
4. **What We Predict or Rank (Label/Proxy):** Rank assets by the probability of ongoing search decay ($	ext{is\_declining} = 1$ if `trend_direction == 'down'`), producing an ordered top-50 review queue for human editors.
5. **What We Deliberately Exclude:** Retrospective slope metrics (`trend_pct` and `trend_direction`) because they encode future outcome slope and introduce catastrophic target leakage; all client PII and raw URLs are also strictly excluded.


## 2. Prove Three Facts with Three Queries

We execute three verification queries using DuckDB against the FlyRank dataset slice to prove the grain, counts/spans, and availability.


In [1]:
import duckdb
import pandas as pd
import numpy as np

# Load the dataset slice
df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')
print(f"Loaded slice with {len(df):,} rows and {len(df.columns)} columns.\n")

# --- QUERY 1: PROVE THE GRAIN (Zero duplicate content_ids) ---
print("--- QUERY 1: Grain Verification (Must return 0 duplicate rows) ---")
q1_grain = duckdb.query('''
    SELECT content_id, COUNT(*) as row_count 
    FROM df 
    GROUP BY content_id 
    HAVING COUNT(*) > 1 
    LIMIT 5
''').df()

print(f"Duplicate rows returned: {len(q1_grain)}")
assert len(q1_grain) == 0, "Grain violation: multiple rows found for a single content_id!"
print("Verdict: Grain holds perfectly — exactly 1 row per content_id.\n")

# --- QUERY 2: PROVE ROW COUNT AND DATE SPAN ---
print("--- QUERY 2: Row Count & Date Span ---")
q2_span = duckdb.query('''
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_id) as client_domains,
        MIN(content_age_days) as min_content_age_days,
        MAX(content_age_days) as max_content_age_days,
        MIN(days_since_last_update) as min_days_since_update,
        MAX(days_since_last_update) as max_days_since_update
    FROM df
''').df()
print(q2_span.to_string(index=False))
print("\nVerdict: Slice covers 30,000 rows across 32 clients, with content age spanning 90 to 564 days.\n")

# --- QUERY 3: PROVE AVAILABILITY WITH 'IS TRUE' ---
print("--- QUERY 3: Availability Verification (Filter with IS TRUE) ---")
# Define analytics availability flag (measurable search volume & sessions)
df['analytics_available'] = (df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)

q3_avail = duckdb.query('''
    SELECT 
        COUNT(*) as surviving_rows,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM df), 2) as surviving_pct
    FROM df 
    WHERE analytics_available IS TRUE
''').df()
print(q3_avail.to_string(index=False))
assert q3_avail['surviving_rows'].iloc[0] > 0, "No rows survived availability filter!"
print("Verdict: 22,006 rows (73.35%) survive with full search and analytics availability IS TRUE.")


Loaded slice with 30,000 rows and 44 columns.

--- QUERY 1: Grain Verification (Must return 0 duplicate rows) ---
Duplicate rows returned: 0
Verdict: Grain holds perfectly — exactly 1 row per content_id.

--- QUERY 2: Row Count & Date Span ---
 total_rows  client_domains  min_content_age_days  max_content_age_days  min_days_since_update  max_days_since_update
      30000              32                    90                   564                      1                    373

Verdict: Slice covers 30,000 rows across 32 clients, with content age spanning 90 to 564 days.

--- QUERY 3: Availability Verification (Filter with IS TRUE) ---
 surviving_rows  surviving_pct
          22006          73.35
Verdict: 22,006 rows (73.35%) survive with full search and analytics availability IS TRUE.


## 3. Five Features, Max (With 'Available-When?' Lines)

We construct a tight 5-feature baseline frame strictly observable prior to the editorial decision moment:

1. **`days_since_last_update`**: *Knowable at the decision moment because the CMS records the exact publication timestamp of the most recent editorial edit.*
2. **`log_impressions_90d`**: *Knowable at the decision moment because Google Search Console trailing 90-day search impressions are updated daily.*
3. **`avg_position`**: *Knowable at the decision moment because average organic SERP ranking across queries is reported directly in historical Search Console exports.*
4. **`ctr`**: *Knowable at the decision moment because trailing clicks divided by impressions is computed from historical SERP performance.*
5. **`word_count`**: *Knowable at the decision moment because the published text length of the page is static and inspectable via the CMS content body.*


In [2]:
# Construct the 5-feature frame
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
FIVE_FEATURES = ['days_since_last_update', 'log_impressions_90d', 'avg_position', 'ctr', 'word_count']

# Fill missing word counts with 0 (thin/media assets)
feature_frame = df[FIVE_FEATURES].fillna(0)
print(f"Constructed 5-Feature Frame: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]} features")
print("\nFeature Summary Statistics:")
print(feature_frame.describe().T[['mean', 'std', 'min', '50%', 'max']])


Constructed 5-Feature Frame: 30,000 rows x 5 features

Feature Summary Statistics:
                               mean          std       min          50%  \
days_since_last_update    46.098300    42.078709  1.000000    20.000000   
log_impressions_90d        6.188688     2.688539  0.693147     6.595781   
avg_position              16.342380    15.216790  0.000000    10.800000   
ctr                        0.510733     3.279162  0.000000     0.070000   
word_count              2310.205433  1846.788556  0.000000  2605.000000   

                                max  
days_since_last_update   373.000000  
log_impressions_90d       13.157182  
avg_position             245.000000  
ctr                      100.000000  
word_count              9546.000000  


## 4. The Trap: Springing and Removing Target Leakage

In search analytics datasets, retrospective slope indicators like `trend_pct` mathematically derive from the outcome window. 
Because `is_declining = 1` is defined by `trend_direction == 'down'`, and `trend_direction` is calculated from `trend_pct < -0.10`, adding `trend_pct` leaks the answer key.

We deliberately perform this experiment:
1. **Model A (Leaky):** Train on 5 features **plus `trend_pct`**. Watch the precision score jump to a "too good to be true" 1.0000.
2. **Model B (Honest):** Remove `trend_pct`, retrain, and keep the honest, leak-free baseline score.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

y = (df['trend_direction'] == 'down').astype(int)

# Grouped Client Split (80/20)
clients = df['client_id'].unique()
np.random.seed(42)
test_clients = set(np.random.choice(clients, size=int(len(clients) * 0.2), replace=False))
test_mask = df['client_id'].isin(test_clients)

# --- PART 1: SPRING THE TRAP (Add trend_pct) ---
X_leaky = df[FIVE_FEATURES + ['trend_pct']].fillna(0)
rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_leaky.fit(X_leaky[~test_mask], y[~test_mask])

leaky_scores = rf_leaky.predict_proba(X_leaky[test_mask])[:, 1]
top50_leaky_idx = np.argsort(-leaky_scores)[:50]
leaky_p50 = y[test_mask].iloc[top50_leaky_idx].mean()

# --- PART 2: REMOVE THE LEAK (Honest 5 Features) ---
X_honest = df[FIVE_FEATURES].fillna(0)
rf_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_honest.fit(X_honest[~test_mask], y[~test_mask])

honest_scores = rf_honest.predict_proba(X_honest[test_mask])[:, 1]
top50_honest_idx = np.argsort(-honest_scores)[:50]
honest_p50 = y[test_mask].iloc[top50_honest_idx].mean()

print("="*65)
print("             THE TARGET LEAKAGE TRAP EXPERIMENT                 ")
print("="*65)
print(f"Model A (With 'trend_pct' LEAK)  Precision@50 : {leaky_p50:.4f}  (Artificially Perfect!)")
print(f"Model B (Honest 5-Feature Model) Precision@50 : {honest_p50:.4f}  (Real-World Holdout)")
print("="*65)
print("Observation: When trend_pct is included, the model simply memorizes the target threshold.")
print("Resolution: trend_pct is permanently deleted from candidate features to ensure an honest pipeline.")


             THE TARGET LEAKAGE TRAP EXPERIMENT                 
Model A (With 'trend_pct' LEAK)  Precision@50 : 1.0000  (Artificially Perfect!)
Model B (Honest 5-Feature Model) Precision@50 : 0.4800  (Real-World Holdout)
Observation: When trend_pct is included, the model simply memorizes the target threshold.
Resolution: trend_pct is permanently deleted from candidate features to ensure an honest pipeline.


## 5. Named Limitations of this Slice

- **Observational, Not Causal:** This dataset records historical page performance; it cannot prove that rewriting an asset will causally cause Google to restore traffic. Causal efficacy requires randomized A/B experimentation.
- **Unobserved Macro SERP Shocks:** Page-level metrics do not record external SERP layout disruptions (Google AI Overviews, rich snippet takeovers, broad core updates).
- **History Imbalance:** Certain client domains joined recently and have shorter historical baselines than older clients.

---

## 6. Self-Check

- [x] Five plain-words contract answers provided in Section 1
- [x] Exactly three verification queries with visible outputs in Section 2 (availability checked with `IS TRUE`)
- [x] Five-feature frame with an "available when?" rationale per feature in Section 3
- [x] Deliberate-leak experiment shown (1.0000 vs. honest) and removed in Section 4
- [x] Named limitations documented in Section 5
